In [5]:
import os
from pathlib import Path
from dotenv import load_dotenv

from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage

load_dotenv(Path(r"C:/LLM agent/Aiprojects/.env"))
api_key = os.getenv("OPENAI_API_KEY")

model = ChatOpenAI(model="gpt-4o-mini", openai_api_key=api_key)

store = {}

def get_session_history(session_id: str):
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]

with_message_history = RunnableWithMessageHistory(model, get_session_history)


In [6]:
config = {"configurable": {"session_id": "abc2"}}  # 세션 ID를 설정하는 config 객체 생성
# 현재 세션 id가 abc2

response = with_message_history.invoke(
    [HumanMessage(content="안녕? 난 임효정이야.")],
    config=config,
)

print(response.content)

안녕하세요, 임효정님! 만나서 반갑습니다. 어떻게 도와드릴까요?


In [7]:
response = with_message_history.invoke(
    [HumanMessage(content="내 이름이 뭐지?")],
    config=config,
)

print(response.content)

당신의 이름은 임효정입니다. 다른 질문이 있으신가요?


In [8]:
config = {"configurable": {"session_id": "abc3"}}
# 현재 세션 id를 abc2 -> abc3으로 변경
response = with_message_history.invoke(
    [HumanMessage(content="내 이름이 뭐지?")],
    config=config,
)

response.content

'죄송하지만, 당신의 이름을 알 수 있는 방법이 없습니다. 당신의 이름을 알려주시면 그에 맞춰 대화할 수 있습니다!'

In [9]:
config = {"configurable": {"session_id": "abc2"}}

response = with_message_history.invoke(
    [HumanMessage(content="아까 우리가 무슨 얘기 했지?")],
    config=config,
)

response.content

'우리는 당신의 이름이 임효정이라는 것에 대해 이야기했습니다. 더 궁금한 점이나 다른 주제로 이야기하고 싶은 것이 있다면 말씀해 주세요!'

In [10]:
config = {"configurable": {"session_id": "abc2"}}
for r in with_message_history.stream(
    [HumanMessage(content = "내가 어느 나라 사람인지 맞춰보고, 그 나라의 문화에 대해 말해봐")],
    config=config,
):
    print(r.content, end="|")
# 여기서 |로 토큰 값을 확인 해보자.

# |정보|가| 부족|하지만| "|김|기|원|"|이라는| 이름|으로| 미|루|어|보|아| 한국|에서| 오|셨|을| 가능|성이| 높|습니다|.| 맞|나요|?| 
# |한국|은| 풍|부|한| 역사|와| 문|화를| 가지고| 있습니다|.| 예|를| 들어|:
# |1|.| **|전|통| 음식|**|:| 김|치|,| 불|고|기|,| 비|빔|밥| 등| 한국|음|식|이| 세계|적으로| 유명|합니다|.| 특히| 김|치는| 건강|에| 좋|고| 다양한| 재|료|로| 만들어|져| 있습니다|.
# |2|.| **|명|절|과| 기|념|일|**|:| 설|날|(|한국|의| 새|해|)|과| 추|석|(|가|을|의| 수|확|을| 기|념|하는| 명|절|)은| 가족|들이| 모|여|서| 서로|의| 안|부|를| 묻|고|,| 전|통| 음|식을| 나|누|는| 중요한| 날|입니다|.
# |3|.| **|한|복|**|:| 한국|의| 전|통| 의|상|으로|,| 결|혼|식|이나| 명|절| 같은| 특별|한| 날|에| 입|습니다|.| 화|려|하고| 다양|하게| 디자인|되어| 있습니다|.
# |4|.| **|K|-P|op|과| 드|라마|**|:| 최근| 몇| 년| 간| K|-P|op|과| 한국| 드|라마|가| 글로벌|한| 인|기를| 얻|으|면서| 한국| 문화|도| 많은| 나라|에서| 사랑|받|고| 있습니다|.
# |더| 궁|금|한| 내용|이| 있|으면| 말씀|해| 주세요|!||||

|임|효|정|이라는| 이름|으로| 보|아| 한국|에서| 오|신| 것|일| 가능|성이| 높|습니다|.| 한국|의| 문화|에| 대해| 간|단|히| 설명|해| 드|릴|게|요|.

|한국|은| 오|랜| 역사|와| 독|특|한| 문|화를| 가진| 나라|입니다|.| 한국|의| 전|통| 문화|에는| 한|복|,| 김|치|,| 그리고| 다양한| 전|통| 음악|과| 춤|이| 포함|됩니다|.| 또한|,| 한국|은| K|-|팝|,| 드|라마|,| 영화| 등| 현대| 대|중|문화|로|도| 세계|적인| 인|기를| 끌|고| 있습니다|.

|한국|의| 음식|은| 매우| 다양|하고| 맛|있|습니다|.| 대표|적인| 음식|으로|는| 비|빔|밥|,| 불|고|기|,| 떡|볶|이| 등이| 있습니다|.| 또한|,| 한국|의| 명|절|인| 설|날|과| 추|석|은| 가족|과| 함께|하는| 중요한| 시간|으로|,| 전|통|적인| 음식|과| 의|례|가| 이루|어|집|니다|.

|한국|의| 전|통| 예|술|도| 매|력|적|입니다|.| 예|를| 들어|,| 한국|의| 서|예|와| 도|자|기|,| 그리고| 전|통| 음악|인| 국|악|은| 독|특|한| 미|적| 가|치를| 가지고| 있습니다|.

|더| 알고| 싶은| 한국|의| 문화|나| 다른| 주|제가| 있다|면| 말씀|해| 주세요|!||